Set custom plugin dir path :

In [ ]:
import os
 
os.environ["FIFTYONE_PLUGINS_DIR"] = "./voxel_visualization" #Set path to voxel plugins directory

Run Voxel Session:

In [ ]:
import fiftyone as fo
import json
import pandas as pd

# Paths
images_path =  #Path to folder containing crops
metadata_path =  #Path to csv file containing attributes for objects
results_path =  #LLM Inference Results

# Create a FiftyOne dataset
dataset = fo.Dataset("Vehicle_Dataset")

# Define relevant fields
FIELDS = {"type", "occlusion", "truncation", "vehicleType", "hasOversizedCargo", "isBehindPhysicalBarrier", "isBrakeLightsOn", "isCargo", "isDoorOpen", "isDummy", "isFlashingLightsOn", "isIgnoreArea", "isReverseLightsOn", "isTurnLightIndicatorLeftOn", "isTurnLightIndicatorRightOn", "isVehicleLightsOn", "isWarnLightsOn"}

# Parse metadata CSV
def parse_metadata(metadata_file_path):
    df = pd.read_csv(metadata_file_path)
    return {
        f"{int(row['index'])}.png": {key: str(row[key]) for key in FIELDS if key in row}
        for _, row in df.iterrows()
    }

# Parse results JSON
def parse_results(results_file_path):
    with open(results_file_path, "r") as f:
        results_data = json.load(f)
    return {
        entry["image_name"]: {key: str(value) for key, value in entry.get("Vehicle", {}).items()}
        for entry in results_data if "image_name" in entry
    }

# Load metadata and results
ground_truths = parse_metadata(metadata_path)
predictions = parse_results(results_path)

# Add samples to dataset
for img_id, gt_attributes in ground_truths.items():
    gt_filepath = os.path.join(images_path, img_id)
    sample = fo.Sample(filepath=gt_filepath)
    
    # Assign ground truth and predicted attributes
    for key in FIELDS:
        gt_value = gt_attributes.get(key, "Unknown")
        pred_value = predictions.get(img_id, {}).get(key, "Unknown")
        
        sample[f"{key}_gt"] = gt_value
        sample[f"{key}_pred"] = f'Pred: {pred_value}'
        sample[f"{key}_match"] = "PASS" if gt_value == pred_value else "FAIL"
    
    dataset.add_sample(sample)

# Launch FiftyOne session
fo.launch_app(dataset)
